<tabla align="centro">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visite el aprendizaje profundo del MIT</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab2/solutions/TF_Part2_Debiasing_Solution.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" />Ejecutar en Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab2/solutions/TF_Part2_Debiasing_Solution.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png" height="70px" style="padding-bottom:5px;"  />Ver código fuente en GitHub</a></td>
</tabla>

# Información de derechos de autor

In [ ]:
# Copyright 2026 MIT 6.S191 Introducción al aprendizaje profundo. Reservados todos los derechos.
# 
# Licenciado bajo la Licencia MIT. No puede utilizar este archivo excepto en cumplimiento
# con la Licencia. El uso y/o modificación de este código fuera de 6.S191 debe
# referencia:
# 
# © MIT 6.S191: Introducción al aprendizaje profundo
# http://intotodeeplearning.com
# 

# Laboratorio 2: Visión por Computador

# Parte 2: Desestabilización de los sistemas de detección facial

En la segunda parte del laboratorio, exploraremos dos aspectos destacados del aprendizaje profundo aplicado: la detección facial y el sesgo algorítmico.

Implementar sistemas de IA justos e imparciales es fundamental para su aceptación a largo plazo. Considere la tarea de detección facial: dada una imagen, ¿es la imagen de un rostro?  Esta tarea aparentemente simple, pero extremadamente importante, está sujeta a cantidades significativas de sesgo algorítmico entre grupos demográficos seleccionados.

En esta práctica de laboratorio, investigaremos [one recently published approach](http://introtodeeplearning.com/AAAI_MitigatingAlgorithmicBias.pdf) para abordar el sesgo algorítmico. Construiremos un modelo de detección facial que aprenda las *variables latentes* subyacentes a los conjuntos de datos de imágenes faciales y los utilice para volver a muestrear de forma adaptativa los datos de entrenamiento, mitigando así cualquier sesgo que pueda estar presente para entrenar un modelo *dessesgado*.


Ejecute el siguiente bloque de código para ver un breve vídeo de Google que explora cómo y por qué es importante tener en cuenta los prejuicios al pensar en el aprendizaje automático:

In [ ]:
import IPython
IPython.display.YouTubeVideo('59bMh59JQDo')

Comencemos instalando las dependencias relevantes.

Usaremos Comet ML para rastrear el desarrollo de nuestro modelo y las ejecuciones de capacitación.

1. Regístrese para obtener una cuenta de Comet: [HERE](https://www.comet.com/signup?utm_source=mit_dl&utm_medium=partner&utm_content=github)
2. Esto generará una clave API personal, que puede encontrar en la primera página 'Comenzar con Comet', en la configuración de su cuenta o presionando el botón '?' en la esquina superior derecha y luego 'Guía de inicio rápido'. Ingrese esta clave API como la variable global `COMET_API_KEY` a continuación.


In [ ]:
!pip install comet_ml --quiet
import comet_ml
# TODO: ¡¡INGRESA TU CLAVE API AQUÍ!! instrucciones arriba
COMET_API_KEY = ""

# Importar Tensorflow 2.0
import tensorflow as tf

import IPython
import functools
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Descargue e importe el paquete MIT 6.S191
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# Comprobar que estamos usando una GPU, si no cambiar de tiempo de ejecución
# usando Runtime > Cambiar tipo de tiempo de ejecución > GPU
assert len(tf.config.list_physical_devices('GPU')) > 0
assert COMET_API_KEY != "", "Please insert your Comet API Key"

## 2.1 Conjuntos de datos

Usaremos tres conjuntos de datos en esta práctica de laboratorio. Para entrenar nuestros modelos de detección facial, necesitaremos un conjunto de datos de ejemplos positivos (es decir, de caras) y un conjunto de datos de ejemplos negativos (es decir, de cosas que no son caras). Usaremos estos datos para entrenar nuestros modelos para clasificar imágenes como caras o no caras. Finalmente, necesitaremos un conjunto de datos de prueba de imágenes de rostros. Dado que nos preocupa el posible *sesgo* de nuestros modelos aprendidos frente a ciertos datos demográficos, es importante que el conjunto de datos de prueba que utilicemos tenga la misma representación en todos los datos demográficos o características de interés. En este laboratorio, consideraremos el tono de piel y el género.

1. **Datos positivos de entrenamiento**: [CelebA Dataset](http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html). Una gran escala (más de 200.000 imágenes) de rostros de celebridades.   
2. **Datos de entrenamiento negativos**: [ImageNet](http://www.image-net.org/). Muchas imágenes en muchas categorías diferentes. Tomaremos ejemplos negativos de una variedad de categorías no humanas.
[Fitzpatrick Scale](https://en.wikipedia.org/wiki/Fitzpatrick_scale) sistema de clasificación de tipos de piel, con cada imagen etiquetada como "Más clara" u "Más oscura".

Comencemos importando estos conjuntos de datos. Hemos escrito una clase que realiza un poco de preprocesamiento de datos para importar los datos de entrenamiento en un formato utilizable.

In [ ]:
# Obtenga los datos de entrenamiento: ambas imágenes de CelebA e ImageNet
path_to_training_data = tf.keras.utils.get_file('train_face.h5', 'https://www.dropbox.com/s/hlz8atheyozp1yx/train_face.h5?dl=1')
# Crear una instancia de TrainingDatasetLoader utilizando el conjunto de datos descargado
loader = mdl.lab2.TrainingDatasetLoader(path_to_training_data)

Podemos observar el tamaño del conjunto de datos de entrenamiento y tomar un lote de tamaño 100:

In [ ]:
number_of_training_examples = loader.get_train_size()
(images, labels) = loader.get_batch(100)

¡Juegue mostrando imágenes para tener una idea de cómo se ven realmente los datos de entrenamiento!

In [ ]:
# ## Examinando el conjunto de datos de entrenamiento de CelebA ###

# @title ¡Cambie los controles deslizantes para ver ejemplos de entrenamiento positivos y negativos! {ejecutar: "automático" }

face_images = images[np.where(labels==1)[0]]
not_face_images = images[np.where(labels==0)[0]]

idx_face = 23 # @param {tipo:"control deslizante", min:0, max:50, paso:1}
idx_not_face = 9 # @param {tipo:"control deslizante", min:0, max:50, paso:1}

plt.figure(figsize=(5,5))
plt.subplot(1, 2, 1)
plt.imshow(face_images[idx_face])
plt.title("Face"); plt.grid(False)

plt.subplot(1, 2, 2)
plt.imshow(not_face_images[idx_not_face])
plt.title("Not Face"); plt.grid(False)

### Pensando en el sesgo

Recuerde que entrenaremos nuestros clasificadores de detección facial en el conjunto de datos grande y bien seleccionado de CelebA (e ImageNet) y luego evaluaremos su precisión probándolos en un conjunto de datos de prueba independiente. Nuestro objetivo es construir un modelo que se entrene en CelebA *y* logre una alta precisión de clasificación en el conjunto de datos de prueba en todos los datos demográficos y, por lo tanto, mostrar que este modelo no sufre ningún sesgo oculto.

¿Qué queremos decir exactamente cuando decimos que un clasificador está sesgado? Para formalizar esto, necesitaremos pensar en [*latent variables*](https://en.wikipedia.org/wiki/Latent_variable), variables que definen un conjunto de datos pero que no se observan estrictamente. Como se definió en la lección sobre modelado generativo, usaremos el término *espacio latente* para referirnos a las distribuciones de probabilidad de las variables latentes antes mencionadas. Al juntar estas ideas, consideramos que un clasificador está *sesgado* si su decisión de clasificación cambia después de ver algunas características latentes adicionales. Puede ser útil tener presente esta noción de sesgo durante el resto del laboratorio.

## 2.2 CNN para detección facial

Primero, definiremos y entrenaremos una CNN en la tarea de clasificación facial y evaluaremos su precisión. Más adelante, evaluaremos el rendimiento de nuestros modelos dessesgados con respecto a esta CNN de referencia. El modelo CNN tiene una arquitectura relativamente estándar que consta de una serie de capas convolucionales con normalización por lotes seguidas de dos capas completamente conectadas para aplanar la salida de convolución y generar una predicción de clase.

### Definir y entrenar el modelo CNN

Como hicimos en la primera parte del laboratorio, definiremos nuestro modelo CNN y luego entrenaremos en los conjuntos de datos de CelebA e ImageNet usando la clase `tf.GradientTape` y el método `tf.GradientTape.gradient`.

In [ ]:
# ## Definir el modelo CNN ###

n_filters = 12 # número base de filtros convolucionales

'''Function to define a standard CNN model'''
def make_standard_classifier(n_outputs=1):
  Conv2D = functools.partial(tf.keras.layers.Conv2D, padding='same', activation='relu')
  BatchNormalization = tf.keras.layers.BatchNormalization
  Flatten = tf.keras.layers.Flatten
  Dense = functools.partial(tf.keras.layers.Dense, activation='relu')

  model = tf.keras.Sequential([
    Conv2D(filters=1*n_filters, kernel_size=5,  strides=2),
    BatchNormalization(),

    Conv2D(filters=2*n_filters, kernel_size=5,  strides=2),
    BatchNormalization(),

    Conv2D(filters=4*n_filters, kernel_size=3,  strides=2),
    BatchNormalization(),

    Conv2D(filters=6*n_filters, kernel_size=3,  strides=2),
    BatchNormalization(),

    Flatten(),
    Dense(512),
    Dense(n_outputs, activation=None),
  ])
  return model

standard_classifier = make_standard_classifier()

¡Ahora entrenemos la CNN estándar!

In [ ]:
# ## Crear un experimento Comet para realizar un seguimiento de nuestra ejecución de entrenamiento ###
def create_experiment(project_name, params):
  # finalizar cualquier experimento previo
  if 'experiment' in locals():
    experiment.end()

  # iniciar el experimento del cometa para el seguimiento
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name=project_name)
  # Registre nuestros hiperparámetros, definidos anteriormente, en el experimento.
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment


In [ ]:
# ## Entrena la CNN estándar ###

# Hiperparámetros de entrenamiento
params = dict(
  batch_size = 32,
  num_epochs = 2,  # mantente pequeño para correr más rápido
  learning_rate = 5e-4,
)

experiment = create_experiment("6S191_Lab2_Part2_CNN", params)

optimizer = tf.keras.optimizers.Adam(params["learning_rate"]) # definir nuestro optimizador
loss_history = mdl.util.LossHistory(smoothing_factor=0.99) # para registrar la evolución de las pérdidas
plotter = mdl.util.PeriodicPlotter(sec=2, scale='semilogy')
if hasattr(tqdm, '_instances'): tqdm._instances.clear() # claro si existe

@tf.function
def standard_train_step(x, y):
  with tf.GradientTape() as tape:
    # introducir las imágenes en el modelo
    logits = standard_classifier(x)
    # Calcular la perdida
    loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits)

  # Propagación hacia atrás
  grads = tape.gradient(loss, standard_classifier.trainable_variables)
  optimizer.apply_gradients(zip(grads, standard_classifier.trainable_variables))
  return loss

# ¡El circuito de entrenamiento!
step = 0
for epoch in range(params["num_epochs"]):
  for idx in tqdm(range(loader.get_train_size()//params["batch_size"])):
    # Obtenga un lote de datos de entrenamiento y propáguelos a través de la red
    x, y = loader.get_batch(params["batch_size"])
    loss = standard_train_step(x, y)

    # Registrar la pérdida y trazar la evolución de la pérdida en función del entrenamiento.
    loss_history.append(loss.numpy().mean())
    plotter.plot(loss_history.get())

    experiment.log_metric("loss", loss.numpy().mean(), step=step)
    step += 1

### Evaluar el rendimiento de la CNN estándar

A continuación, evaluemos el rendimiento de clasificación de nuestra CNN estándar entrenada con CelebA en el conjunto de datos de entrenamiento.


In [ ]:
# ## Evaluación de CNN estándar ###

# DATOS DE ENTRENAMIENTO
# Evaluar en un subconjunto de CelebA+Imagenet
(batch_x, batch_y) = loader.get_batch(5000)
y_pred_standard = tf.round(tf.nn.sigmoid(standard_classifier.predict(batch_x)))
acc_standard = tf.reduce_mean(tf.cast(tf.equal(batch_y, y_pred_standard), tf.float32))

print("Standard CNN accuracy on (potentially biased) training set: {:.4f}".format(acc_standard.numpy()))

También evaluaremos nuestras redes en un conjunto de datos de prueba independiente que contiene caras que no se vieron durante el entrenamiento. Para los datos de la prueba, analizaremos la precisión de la clasificación en cuatro grupos demográficos diferentes, según la escala de piel de Fitzpatrick y las etiquetas basadas en el sexo: hombre de piel oscura, mujer de piel oscura, hombre de piel clara y mujer de piel clara.

Echemos un vistazo a algunas caras de muestra en el conjunto de prueba.

In [ ]:
# ## Cargar conjunto de datos de prueba y ejemplos de trazado ###

test_faces = mdl.lab2.get_test_faces()
keys = ["Light Female", "Light Male", "Dark Female", "Dark Male"]
for group, key in zip(test_faces,keys):
  plt.figure(figsize=(5,5))
  plt.imshow(np.hstack(group))
  plt.title(key, fontsize=15)

Ahora, evaluemos la probabilidad de que cada uno de estos datos demográficos faciales se clasifique como rostro utilizando el clasificador CNN estándar que acabamos de entrenar.

In [ ]:
# ## Evalúe la CNN estándar en los datos de prueba ###

standard_classifier_logits = [standard_classifier(np.array(x, dtype=np.float32)) for x in test_faces]
standard_classifier_probs = tf.squeeze(tf.sigmoid(standard_classifier_logits))

# Trazar las precisiones de predicción por grupo demográfico
xx = range(len(keys))
yy = standard_classifier_probs.numpy().mean(1)
plt.bar(xx, yy)
plt.xticks(xx, keys)
plt.ylim(max(0,yy.min()-np.ptp(yy)/2.), yy.max()+np.ptp(yy)/2.)
plt.title("Standard classifier predictions");

Eche un vistazo a las precisiones de este primer modelo en estos cuatro grupos. ¿Qué observas? ¿Consideraría este modelo sesgado o imparcial? ¿Cuáles son algunas de las razones por las que un modelo entrenado puede tener precisiones sesgadas?

## 2.3 Mitigar el sesgo algorítmico

Los desequilibrios en los datos de entrenamiento pueden provocar un sesgo algorítmico no deseado. Por ejemplo, la mayoría de los rostros en CelebA (nuestro conjunto de entrenamiento) son de mujeres de piel clara. Como resultado, un clasificador entrenado en CelebA será más adecuado para reconocer y clasificar rostros con características similares a estas y, por lo tanto, estará sesgado.

¿Cómo podríamos superar esto? Una solución ingenua (y que están adoptando muchas empresas y organizaciones) sería anotar diferentes subclases (es decir, mujeres de piel clara, hombres con sombreros, etc.) dentro de los datos de entrenamiento y luego igualar manualmente los datos con respecto a estos grupos.

Pero este enfoque tiene dos desventajas importantes. Primero, requiere anotar cantidades masivas de datos, lo cual no es escalable. En segundo lugar, requiere que sepamos qué sesgos potenciales (por ejemplo, raza, género, pose, oclusión, sombreros, gafas, etc.) buscar en los datos. Como resultado, es posible que la anotación manual no capture todas las diferentes características que están desequilibradas dentro de los datos de entrenamiento.

En lugar de eso, **aprendamos** estas características de manera imparcial y no supervisada, sin necesidad de ninguna anotación, y luego entrenemos a un clasificador de manera justa con respecto a estas características. En el resto de este laboratorio, haremos exactamente eso.

## 2.4 Autocodificador variacional (VAE) para aprender la estructura latente

Como vio, la precisión de la CNN varía según los cuatro grupos demográficos que analizamos. Para pensar por qué puede ser esto, considere el conjunto de datos en el que se entrenó el modelo, CelebA. Si ciertas características, como la piel oscura o los sombreros, son *raras* en CelebA, el modelo puede terminar sesgado en contra de ellas como resultado del entrenamiento con un conjunto de datos sesgado. Es decir, su precisión de clasificación será peor en rostros que tienen rasgos subrepresentados, como rostros de piel oscura o rostros con sombreros, en comparación con rostros con rasgos bien representados en los datos de entrenamiento. Esto es un problema.

Nuestro objetivo es entrenar una versión *desesgada* de este clasificador, una que tenga en cuenta las posibles disparidades en la representación de características dentro de los datos de entrenamiento. Específicamente, para construir un clasificador facial dessesgado, entrenaremos un modelo que **aprenda una representación del espacio latente subyacente** a los datos de entrenamiento facial. Luego, el modelo utiliza esta información para mitigar sesgos no deseados al muestrear rostros con características raras, como piel oscura o sombreros, *con mayor frecuencia* durante el entrenamiento. El requisito de diseño clave para nuestro modelo es que pueda aprender una *codificación* de las características latentes en los datos faciales de una manera completamente *sin supervisión*. Para lograr esto, recurriremos a codificadores automáticos variacionales (VAE).

![The concept of a VAE](https://i.ibb.co/3s4S6Gc/vae.jpg)

Como se muestra en el esquema anterior y en la Conferencia 4, los VAE se basan en una estructura de codificador-decodificador para aprender una representación latente de los datos de entrada. En el contexto de la visión por computadora, la red codificadora toma imágenes de entrada, las codifica en una serie de variables definidas por una media y una desviación estándar, y luego extrae de las distribuciones definidas por estos parámetros para generar un conjunto de variables latentes muestreadas. Luego, la red decodificadora "decodifica" estas variables para generar una reconstrucción de la imagen original, que se utiliza durante el entrenamiento para ayudar al modelo a identificar qué variables latentes es importante aprender.

Formalicemos dos aspectos clave del modelo VAE y definamos funciones relevantes para cada uno.


### Comprender los VAE: función de pérdida

En la práctica, ¿cómo podemos entrenar un VAE? Al aprender el espacio latente, restringimos las medias y las desviaciones estándar para que sigan aproximadamente una unidad gaussiana. Recuerde que estos son parámetros aprendidos y, por lo tanto, deben tenerse en cuenta en el cálculo de la pérdida, y que la parte del decodificador del VAE utiliza estos parámetros para generar una reconstrucción que debe coincidir estrechamente con la imagen de entrada, que también debe tenerse en cuenta en la pérdida. Lo que esto significa es que tendremos dos términos en nuestra función de pérdida VAE:

1. **Pérdida latente ($L_{KL}$)**: mide qué tan cerca coinciden las variables latentes aprendidas con una unidad gaussiana y se define por la divergencia Kullback-Leibler (KL).
2. **Pérdida de reconstrucción ($L_{x}{(x,\hat{x})}$)**: mide con qué precisión las salidas reconstruidas coinciden con la entrada y está dada por la norma $L^1$ de la imagen de entrada y su salida reconstruida.

La ecuación para la pérdida latente viene dada por:

$$L_{KL}(\mu, \sigma) = \frac{1}{2}\sum_{j=0}^{k-1} (\sigma_j + \mu_j^2 - 1 - \log{\sigma_j})$$

La ecuación para la pérdida de reconstrucción viene dada por:

$$L_{x}{(x,\sombrero{x})} = ||x-\sombrero{x}||_1$$

Así, para la pérdida de VAE tenemos:

$$L_{VAE} = c\cdot L_{KL} + L_{x}{(x,\hat{x})}$$

donde $c$ es un coeficiente de ponderación utilizado para la regularización. Ahora estamos listos para definir nuestra función de pérdida VAE:

In [ ]:
# ## Definición de la función de pérdida VAE ###

''' Function to calculate VAE loss given:
      an input x,
      reconstructed output x_recon,
      encoded means mu,
      encoded log of standard deviation logsigma,
      weight parameter for the latent loss kl_weight
'''
def vae_loss_function(x, x_recon, mu, logsigma, kl_weight=0.0005):
  # TODO: Definir la pérdida latente. Tenga en cuenta que esto se da en la ecuación para L_{KL}
  # en el bloque de texto directamente arriba
  latent_loss = 0.5 * tf.reduce_sum(tf.exp(logsigma) + tf.square(mu) - 1.0 - logsigma, axis=1)
  # pérdida_latente = # TODO

  # TODO: Defina la pérdida de reconstrucción como la media absoluta en píxeles
  # diferencia entre la entrada y la reconstrucción. Pista: necesitarás
  # use tf.reduce_mean y proporcione un argumento de eje que especifique qué
  # Dimensiones a reducir. Por ejemplo, las pérdidas por reconstrucción deben promediar
  # sobre las dimensiones de la imagen de alto, ancho y canal.
  # https://www.tensorflow.org/api_docs/python/tf/math/reduce_mean
  reconstruction_loss = tf.reduce_mean(tf.abs(x-x_recon), axis=(1,2,3))
  # pérdida_reconstrucción = # TODO

  # TODO: Defina la pérdida de VAE. Tenga en cuenta que esto se da en la ecuación para L_{VAE}
  # en el bloque de texto directamente arriba
  vae_loss = kl_weight * latent_loss + reconstruction_loss
  # vae_loss = # TODO

  return vae_loss

¡Excelente! Ahora que tenemos una idea más concreta de cómo funcionan los VAE, exploremos cómo podemos aprovechar esta estructura de red para entrenar un clasificador facial *dessesgado*.

### Comprender los VAE: reparametrización

Como recordará de la conferencia, los VAE utilizan un "truco de reparametrización" para muestrear las variables latentes aprendidas. En lugar de que el codificador VAE genere un único vector de números reales para cada variable latente, genera un vector de medias y un vector de desviaciones estándar que están obligados a seguir aproximadamente distribuciones gaussianas. Luego tomamos muestras de las desviaciones estándar y volvemos a agregar la media para generar esto como nuestro vector latente muestreado. Formalizando esto para una variable latente $z$ donde tomamos muestra $\epsilon \sim N(0,(I))$ tenemos:

$$z = \mu + e^{\left(\frac{1}{2} \cdot \log{\Sigma}\right)}\circ \epsilon$$

donde $\mu$ es la media y $\Sigma$ es la matriz de covarianza. Esto es útil porque nos permitirá definir claramente la función de pérdida para VAE, generar variables latentes muestreadas aleatoriamente, lograr una generalización de red mejorada y ** hacer que nuestra red VAE completa sea diferenciable para que pueda entrenarse mediante retropropagación. ¡Muy poderoso!

Definamos una función para implementar la operación de muestreo VAE:

In [ ]:
# ## Reparametrización de VAE ###

"""Reparameterization trick by sampling from an isotropic unit Gaussian.
# Argumentos
    z_mean, z_logsigma (tensor): mean and log of standard deviation of latent distribution (Q(z|X))
# Devoluciones
    z (tensor): sampled latent vector
"""
def sampling(z_mean, z_logsigma):
  # De forma predeterminada, random.normal es "estándar" (es decir, media = 0 y std = 1,0).
  batch, latent_dim = z_mean.shape
  epsilon = tf.random.normal(shape=(batch, latent_dim))

  # TODO: ¡Defina el cálculo de reparametrización!
  # Tenga en cuenta que la ecuación se proporciona en el bloque de texto inmediatamente superior.
  z = z_mean + tf.math.exp(0.5 * z_logsigma) * epsilon
  # z = # TODO
  return z

## 2.5 Codificador automático variacional sin polarización (DB-VAE)

Ahora, usaremos la idea general detrás de la arquitectura VAE para construir un modelo, denominado [*debiasing variational autoencoder*](https://lmrt.mit.edu/sites/default/files/AIES-19_paper_220.pdf) o DB-VAE, para mitigar los sesgos (potencialmente) desconocidos presentes dentro de la idea de entrenamiento. Entrenaremos nuestro modelo DB-VAE en la tarea de detección facial, ejecutaremos la operación de eliminación de sesgos durante el entrenamiento, evaluaremos el conjunto de datos PPB y compararemos su precisión con nuestro modelo CNN sesgado original.    

### El modelo DB-VAE

La idea clave detrás de este enfoque de eliminación de sesgos es utilizar las variables latentes aprendidas a través de un VAE para volver a muestrear de forma adaptativa los datos de CelebA durante el entrenamiento. Específicamente, alteraremos la probabilidad de que se use una imagen determinada durante el entrenamiento en función de la frecuencia con la que aparecen sus características latentes en el conjunto de datos. Por lo tanto, las caras con rasgos más raros (como piel oscura, gafas de sol o sombreros) deberían tener más probabilidades de ser muestreadas durante el entrenamiento, mientras que la probabilidad de muestreo de rostros con características que están sobrerrepresentadas en el conjunto de datos de entrenamiento debería disminuir (en relación con el muestreo aleatorio uniforme en todos los datos de entrenamiento).

A continuación se muestra un esquema general del enfoque DB-VAE:

![DB-VAE](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/2019/lab2/img/DB-VAE.png)

Recuerde que queremos aplicar nuestro DB-VAE a un problema de *clasificación supervisada*: la tarea de detección facial. Es importante destacar que la parte del codificador en la arquitectura DB-VAE también genera una única variable supervisada, $z_o$, correspondiente a la predicción de clase: cara o no. ¡Por lo general, los VAE no están entrenados para generar variables supervisadas (como una predicción de clase)! Ésta es otra distinción clave entre el DB-VAE y un VAE tradicional.

Tenga en cuenta que solo queremos aprender la representación latente de *caras*, ya que eso es contra lo que en última instancia nos estamos desviando, aunque estemos entrenando un modelo en un problema de clasificación binaria. Necesitaremos asegurarnos de que, **para caras**, nuestro modelo DB-VAE aprenda una representación de las variables latentes no supervisadas, capturadas por la distribución $q_\phi(z|x)$, **y** genere una predicción de clase supervisada $z_o$, pero que, **para ejemplos negativos**, solo genere una predicción de clase $z_o$.

### Definición de la función de pérdida DB-VAE

Esto significa que tendremos que ser un poco inteligentes con la función de pérdida del DB-VAE. La forma de la pérdida dependerá de si lo que se está considerando es una imagen de un rostro o una imagen que no es un rostro.

Para **imágenes de rostros**, nuestra función de pérdida tendrá dos componentes:


1. **Pérdida VAE ($L_{VAE}$)**: consiste en la pérdida latente y la pérdida por reconstrucción.
2. **Pérdida de clasificación ($L_y(y,\hat{y})$)**: pérdida de entropía cruzada estándar para un problema de clasificación binaria.

Por el contrario, para imágenes de **no rostros**, nuestra función de pérdida es únicamente la pérdida de clasificación.

Podemos escribir una única expresión para la pérdida definiendo una variable indicadora ${I}_f$ que refleja qué datos de entrenamiento son imágenes de caras (${I}_f(y) = 1$) y cuáles son imágenes de no caras (${I}_f(y) = 0$). Usando esto obtenemos:

$$L_{total} = L_y(y,\hat{y}) + {I}_f(y)\Big[L_{VAE}\Big]$$

Escribamos una función para definir la función de pérdida DB-VAE:


In [ ]:
# ## Función de pérdida para DB-VAE ###

"""Loss function for DB-VAE.
# Argumentos
    x: true input x
    x_pred: reconstructed x
    y: true label (face or not face)
    y_logit: predicted labels
    mu: mean of latent distribution (Q(z|X))
    logsigma: log of standard deviation of latent distribution (Q(z|X))
# Devoluciones
    total_loss: DB-VAE total loss
    classification_loss = DB-VAE classification loss
"""
def debiasing_loss_function(x, x_pred, y, y_logit, mu, logsigma):

  # TODO: llame a la función relevante para obtener la pérdida de VAE
  vae_loss = vae_loss_function(x, x_pred, mu, logsigma)
  # Foot_loss = foot_loss_function(''TTODO'') # TODO

  # TODO: definir la pérdida de clasificación usando sigmoid_cross_entropy
  # https://www.tensorflow.org/api_docs/python/tf/nn/sigmoid_cross_entropy_with_logits
  classification_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=y_logit)
  # pérdida_clasificación = # TODO

  # Utilice las etiquetas de datos de entrenamiento para crear la variable face_indicator:
  # Indicador que refleja qué datos de entrenamiento son imágenes de caras.
  face_indicator = tf.cast(tf.equal(y, 1), tf.float32)

  # TODO: ¡defina la pérdida total de DB-VAE! Utilice tf.reduce_mean para promediar todo
  # muestras
  total_loss = tf.reduce_mean(
      classification_loss +
      face_indicator * vae_loss
  )
  # total_loss = # TODO

  return total_loss, classification_loss

### Arquitectura DB-VAE

Ahora estamos listos para definir la arquitectura DB-VAE. Para construir DB-VAE, usaremos el clasificador CNN estándar anterior como nuestro codificador y luego definiremos una red de decodificador. Crearemos e inicializaremos los dos modelos y luego construiremos el VAE de un extremo a otro. Usaremos un espacio latente con 100 variables latentes.

La red decodificadora tomará como entrada las variables latentes muestreadas, las ejecutará a través de una serie de capas deconvolucionales y generará una reconstrucción de la imagen de entrada original.

In [ ]:
# ## Definir la parte del decodificador del DB-VAE ###

n_filters = 12 # Número base de filtros convolucionales, igual que CNN estándar.
latent_dim = 100 # número de variables latentes

def make_face_decoder_network():
  # Definir funcionalmente los diferentes tipos de capas que usaremos
  Conv2DTranspose = functools.partial(tf.keras.layers.Conv2DTranspose, padding='same', activation='relu')
  BatchNormalization = tf.keras.layers.BatchNormalization
  Flatten = tf.keras.layers.Flatten
  Dense = functools.partial(tf.keras.layers.Dense, activation='relu')
  Reshape = tf.keras.layers.Reshape

  # Construya la red de decodificadores usando la API secuencial
  decoder = tf.keras.Sequential([
    # Transformar a generación preconvolucional
    Dense(units=4*4*6*n_filters),  # Mapas de características 4x4 (con 6N apariciones)
    Reshape(target_shape=(4, 4, 6*n_filters)),

    # Ampliación de convoluciones (inversa del codificador)
    Conv2DTranspose(filters=4*n_filters, kernel_size=3,  strides=2),
    Conv2DTranspose(filters=2*n_filters, kernel_size=3,  strides=2),
    Conv2DTranspose(filters=1*n_filters, kernel_size=5,  strides=2),
    Conv2DTranspose(filters=3, kernel_size=5,  strides=2),
  ])

  return decoder

Ahora, juntaremos este decodificador con el clasificador CNN estándar como nuestro codificador para definir el DB-VAE. Tenga en cuenta que en este punto, no hay nada especial en la forma en que armamos el modelo que lo convierta en un modelo "dessesgado"; eso vendrá cuando definamos la operación de entrenamiento. Aquí, definiremos la arquitectura central de VAE subclasificando la clase "Modelo"; definir operaciones de codificación, reparametrización y decodificación; y llamar a la red de extremo a extremo.

In [ ]:
# ## Definición y creación del DB-VAE ###

class DB_VAE(tf.keras.Model):
  def __init__(self, latent_dim):
    super(DB_VAE, self).__init__()
    self.latent_dim = latent_dim

    # Defina el número de salidas para el codificador. Recordemos que tenemos
    # variables latentes `latent_dim`, así como una salida supervisada para el
    # clasificación.
    num_encoder_dims = 2*self.latent_dim + 1

    self.encoder = make_standard_classifier(num_encoder_dims)
    self.decoder = make_face_decoder_network()

  # función para introducir imágenes en el codificador, codificar el espacio latente y generar
  # probabilidad de clasificación
  def encode(self, x):
    # salida del codificador
    encoder_output = self.encoder(x)

    # predicción de clasificación
    y_logit = tf.expand_dims(encoder_output[:, 0], -1)
    # parámetros de distribución de variables latentes
    z_mean = encoder_output[:, 1:self.latent_dim+1]
    z_logsigma = encoder_output[:, self.latent_dim+1:]

    return y_logit, z_mean, z_logsigma

  # Reparametrización de VAE: dada una media y logsigma, variables latentes de muestra
  def reparameterize(self, z_mean, z_logsigma):
    # TODO: llamar a la función de muestreo definida anteriormente
    z = sampling(z_mean, z_logsigma)
    # z = # TODO
    return z

  # Decodificar el espacio latente y reconstruir la salida.
  def decode(self, z):
    # TODO: use el decodificador para generar la reconstrucción
    reconstruction = self.decoder(z)
    # reconstruction = # TODO
    return reconstruction

  # La función de llamada se utilizará para pasar las entradas x a través del VAE central.
  def call(self, x):
    # Codificar la entrada en un espacio latente y de predicción.
    y_logit, z_mean, z_logsigma = self.encode(x)

    # TODO: reparameterization
    z = self.reparameterize(z_mean, z_logsigma)
    # z = # TODO

    # TODO: reconstruction
    recon = self.decode(z)
    # recon = # TODO
    return y_logit, z_mean, z_logsigma, recon

  # Predecir cara o no cara logit para una entrada dada x
  def predict(self, x):
    y_logit, z_mean, z_logsigma = self.encode(x)
    return y_logit

dbvae = DB_VAE(latent_dim)

Como se indicó, la arquitectura del codificador es idéntica a la CNN mencionada anteriormente en esta práctica de laboratorio. Tenga en cuenta los resultados de nuestro modelo DB_VAE construido en la función `call`: `y_logit, z_mean, z_logsigma, z`. Piense detenidamente por qué se genera cada uno de estos y su importancia para el problema en cuestión.


### Remuestreo adaptativo para desviación automatizada con DB-VAE

Entonces, ¿cómo podemos utilizar DB-VAE para entrenar un clasificador de detección facial sesgado?

Recuerde la arquitectura DB-VAE: a medida que las imágenes de entrada pasan a través de la red, el codificador aprende una estimación ${Q}(z|X)$ del espacio latente. Queremos aumentar la frecuencia relativa de datos raros mediante un mayor muestreo de regiones subrepresentadas del espacio latente. Podemos aproximar ${Q}(z|X)$ usando las distribuciones de frecuencia de cada una de las variables latentes aprendidas, y luego definir la distribución de probabilidad de seleccionar un punto de datos dado $x$ en función de esta aproximación. Estas distribuciones de probabilidad se utilizarán durante el entrenamiento para volver a muestrear los datos.

Escribirá una función para ejecutar esta actualización de las probabilidades de muestreo y luego llamará a esta función dentro del ciclo de entrenamiento DB-VAE para desviar el modelo.

Primero, hemos definido una breve función auxiliar `get_latent_mu` que devuelve la variable latente media devuelta por el codificador después de que se ingresa un lote de imágenes a la red:

In [ ]:
# Función para devolver los medios para un lote de imágenes de entrada
def get_latent_mu(images, dbvae, batch_size=1024):
  N = images.shape[0]
  mu = np.zeros((N, latent_dim))
  for start_ind in range(0, N, batch_size):
    end_ind = min(start_ind+batch_size, N+1)
    batch = (images[start_ind:end_ind]).astype(np.float32)/255.
    _, batch_mu, _ = dbvae.encode(batch)
    mu[start_ind:end_ind] = batch_mu
  return mu

Ahora, definamos el algoritmo de remuestreo real `get_training_sample_probabilities`. Es importante tener en cuenta el argumento `smoothing_fac`. Este parámetro ajusta el grado de desescalamiento: para `smoothing_fac=0`, el conjunto de entrenamiento remuestreado tenderá a caer uniformemente sobre el espacio latente, es decir, el desescalamiento más extremo.

In [ ]:
# ## Algoritmo de remuestreo para DB-VAE ###

'''Function that recomputes the sampling probabilities for images within a batch
      based on how they distribute across the training data'''
def get_training_sample_probabilities(images, dbvae, bins=10, smoothing_fac=0.001):
    print("Recalcular las probabilidades de muestreo")

    # TODO: ejecutar el lote de entrada y obtener los medios de la variable latente
    mu = get_latent_mu(images, dbvae)
    # mu = get_latent_mu('''TODO''') # TODO

    # probabilidades de muestreo para las imágenes
    training_sample_p = np.zeros(mu.shape[0])

    # Considere la distribución de cada variable latente.
    for i in range(latent_dim):

        latent_distribution = mu[:,i]
        # generar un histograma de la distribución latente
        hist_density, bin_edges =  np.histogram(latent_distribution, density=True, bins=bins)

        # encontrar en qué contenedor latente cae cada muestra de datos
        bin_edges[0] = -float('inf')
        bin_edges[-1] = float('inf')

        # TODO: llame a la función digitalizar para encontrar qué contenedores en la distribución latente
        # cada muestra de datos cae en
        # https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.digitize.html
        bin_idx = np.digitize(latent_distribution, bin_edges)
        # bin_idx = np.digitize('''TODO''', '''TODO''') # TODO

        # suavizar la función de densidad
        hist_smoothed_density = hist_density + smoothing_fac
        hist_smoothed_density = hist_smoothed_density / np.sum(hist_smoothed_density)

        # invertir la función de densidad
        p = 1.0/(hist_smoothed_density[bin_idx-1])

        # TODO: normalizar todas las probabilidades
        p = p / np.sum(p)
        # p = # TODO

        # TODO: actualice las probabilidades de muestreo considerando si el nuevo
        # p calculado es mayor que las probabilidades de muestreo existentes.
        training_sample_p = np.maximum(p, training_sample_p)
        # training_sample_p = # TODO

    # normalización final
    training_sample_p /= np.sum(training_sample_p)

    return training_sample_p

Ahora que hemos definido la actualización de remuestreo, podemos entrenar nuestro modelo DB-VAE con los datos de entrenamiento de CelebA/ImageNet y ejecutar la operación anterior para volver a ponderar la importancia de puntos de datos particulares a medida que entrenamos el modelo. Recuerde nuevamente que solo queremos desviarnos de las características relevantes a las *caras*, no al conjunto de ejemplos negativos. ¡Complete el bloque de código a continuación para ejecutar el ciclo de entrenamiento!

In [ ]:
# ## Entrenando el DB-VAE ###

# Hiperparámetros
params = dict(
  batch_size = 32,
  learning_rate = 5e-4,
  latent_dim = 100,
  num_epochs = 1, # DB-VAE necesita un poco más de épocas para entrenar
)

experiment = create_experiment("6S191_Lab2_Part2_DBVAE", params)

# crear una instancia de un nuevo modelo y optimizador DB-VAE
dbvae = DB_VAE(params["latent_dim"])
optimizer = tf.keras.optimizers.Adam(params["learning_rate"])

# Para definir la operación de entrenamiento, usaremos tf.function que es una herramienta poderosa
# eso nos permite convertir una función de Python en un gráfico de cálculo de TensorFlow.
@tf.function
def debiasing_train_step(x, y):

  with tf.GradientTape() as tape:
    # Introduzca la entrada x en dbvae. Tenga en cuenta que esto utiliza la función de llamada DB_VAE.
    y_logit, z_mean, z_logsigma, x_recon = dbvae(x)

    '''TODO: call the DB_VAE loss function to compute the loss'''
    loss, class_loss = debiasing_loss_function(x, x_recon, y, y_logit, z_mean, z_logsigma)
    # pérdida, class_loss = debiasing_loss_function('''TODO argumentos''') # TODO

  '''TODO: use the GradientTape.gradient method to compute the gradients.
     Hint: this is with respect to the trainable_variables of the dbvae.'''
  grads = tape.gradient(loss, dbvae.trainable_variables)
  # grads = tape.gradient('''TODO''', '''TODO''') # TODO

  # aplicar gradientes a variables
  optimizer.apply_gradients(zip(grads, dbvae.trainable_variables))
  return loss

# obtener caras de entrenamiento del cargador de datos
all_faces = loader.get_all_train_faces()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # claro si existe

# El bucle de entrenamiento: el bucle externo itera sobre el número de épocas
step = 0
for i in range(params["num_epochs"]):

  IPython.display.clear_output(wait=True)
  print("Starting epoch {}/{}".format(i+1, params["num_epochs"]))

  # Recalcular las probabilidades de muestreo de datos.
  '''TODO: recompute the sampling probabilities for debiasing'''
  p_faces = get_training_sample_probabilities(all_faces, dbvae)
  # p_faces = get_training_sample_probabilities('''TODO''', '''TODO''') # TODO

  # obtener un lote de datos de entrenamiento y calcular el paso de entrenamiento
  for j in tqdm(range(loader.get_train_size() // params["batch_size"])):
    # cargar un lote de datos
    (x, y) = loader.get_batch(params["batch_size"], p_pos=p_faces)

    # optimización de pérdidas
    loss = debiasing_train_step(x, y)
    experiment.log_metric("loss", loss.numpy().mean(), step=step)

    # trazar el progreso cada 200 pasos
    if j % 500 == 0:
      mdl.util.plot_sample(x, y, dbvae)

    step += 1

experiment.end()

¡Maravilloso! Ahora deberíamos tener un modelo de clasificación facial entrenado y (¡con suerte!) sesgado, listo para su evaluación!

## 2.6 Evaluación de DB-VAE en un conjunto de datos de prueba

Finalmente, probemos nuestro modelo DB-VAE en el conjunto de datos de prueba, observando específicamente su precisión en cada grupo demográfico de "Hombre oscuro", "Mujer oscura", "Hombre claro" y "Mujer clara". Compararemos el rendimiento de este modelo dessesgado con el estándar CNN (potencialmente sesgado) de anteriormente en el laboratorio.

In [ ]:
dbvae_logits = [dbvae.predict(np.array(x, dtype=np.float32)) for x in test_faces]
dbvae_probs = tf.squeeze(tf.sigmoid(dbvae_logits))

xx = np.arange(len(keys))
plt.bar(xx, standard_classifier_probs.numpy().mean(1), width=0.2, label="Standard CNN")
plt.bar(xx+0.2, dbvae_probs.numpy().mean(1), width=0.2, label="DB-VAE")
plt.xticks(xx, keys);
plt.title("Network predictions on test dataset")
plt.ylabel("Probability"); plt.legend(bbox_to_anchor=(1.04,1), loc="upper left");


## 2.7 Conclusión e información de presentación

Lo alentamos a pensar y tal vez incluso abordar algunas preguntas planteadas por el enfoque y los resultados descritos aquí:

* ¿Cómo se compara la precisión del DB-VAE en los cuatro grupos demográficos con la de la CNN estándar? ¿Le parece sorprendente este resultado de alguna manera?
* ¿Cómo se puede mejorar aún más el rendimiento del clasificador DB-VAE? ¡No optimizamos deliberadamente los hiperparámetros para dejar esto en tus manos!
* ¿En qué aplicaciones (ya sea relacionadas con la detección facial o no) sería deseable eliminar el sesgo de esta manera? ¿Existen aplicaciones en las que quizás no desee desviar su modelo?
* ¿Cree que debería ser necesario que las empresas demuestren que sus modelos, especialmente en el contexto de tareas como la detección facial, no están sesgados? Si es así, ¿tiene alguna idea sobre cómo se podría estandarizar e implementar esto?
* ¿Tiene ideas sobre otras formas de abordar los problemas de sesgo, particularmente en términos de datos de capacitación?

** Intente optimizar su modelo para lograr un mejor rendimiento. Para participar en el concurso, cargue lo siguiente en el sitio de envío de laboratorio para Debiasing Faces Lab ([submission upload link](https://www.dropbox.com/request/SG6AnjrtIljNPrbEOMfP)).**

* Cuaderno Jupyter con el código que usaste para generar tus resultados;
* copia del diagrama de barras de la sección 2.6 que muestra el rendimiento de su modelo;
* una descripción escrita y/o diagrama de la arquitectura y los hiperparámetros que utilizó; si realizó modificaciones adicionales o interesantes al código de la plantilla, inclúyalas en su descripción;
* una discusión escrita de por qué estas modificaciones ayudaron a mejorar el rendimiento.

**Nombre su archivo en el siguiente formato: `[Nombre]_[Apellido]_Cara`, seguido del formato de archivo (.zip, .ipynb, .pdf, etc.).** Se prefieren los archivos ZIP a los archivos individuales. Si envía archivos individuales, debe nombrar los archivos individuales de acuerdo con la nomenclatura anterior (por ejemplo, `[Nombre]_[Apellido]_Cara_TODO.pdf`, `[Nombre]_[Apellido]_Cara_Report.pdf`, etc.).

Esperemos que este laboratorio haya arrojado algo de luz sobre algunos conceptos, desde tareas basadas en la visión hasta VAE y sesgos algorítmicos. Nos gusta pensar que sí, pero somos parciales;).

<img src="https://i.ibb.co/BjLSRMM/ezgif-2-253dfd3f9097.gif" />